In [ ]:
!python -m pip install -U python-woc pandas matplotlib requests
# please clear the output of this cell in your notebook before checking it in

In [1]:
# --- Setup: WoC client and assigned projects ---
import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from tqdm import tqdm
from woc.remote import WocMapsRemote

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

yournetid = 'lwang111'
assignments = pd.read_csv('net2prj.csv')
my_projects = assignments[assignments['netID'] == yournetid].reset_index(drop=True)
my_projects

,netID,WoC,GH
0,lwang111,libatoms_quip,https://github.com/libAtoms/QUIP
1,lwang111,ldbc_sigmod2014-pc,https://github.com/ldbc/sigmod2014-pc
2,lwang111,wurmlab_sequenceserver,https://github.com/wurmlab/sequenceserver
3,lwang111,lanl_elements,https://github.com/lanl/ELEMENTS
4,lwang111,homalg-project_toricvarieties_project,https://github.com/homalg-project/ToricVarieti...
5,lwang111,ph-rast_bmgarch,https://github.com/ph-rast/bmgarch
6,lwang111,nanocomp_meep,https://github.com/NanoComp/meep
7,lwang111,wiris_moodle-filter_wiris,https://github.com/wiris/moodle-filter_wiris
8,lwang111,asgr_profound,https://github.com/asgr/ProFound
9,lwang111,opticspy_lightpipes,https://github.com/opticspy/lightpipes


In [2]:
# --- Step 1: Get commit sha1s for each project (p2c map) ---
# Fetch the list of commit sha1s WoC has for one project.
def fetch_project_commit_shas(woc_id):
    commits = woc.get_values('p2c', woc_id)
    df_project_commits = pd.DataFrame(commits, columns=['sha1'])
    df_project_commits['project'] = woc_id
    print(f'{woc_id}: {len(commits)} commits')
    return df_project_commits

# Reuse a saved sha1 list only if it covers exactly the assigned projects.
def load_cached_commit_shas(path):
    if not os.path.exists(path):
        return None
    cached = pd.read_csv(path)
    covers_all_projects = set(cached['project']) == set(my_projects['WoC'])
    return cached if covers_all_projects else None

df = load_cached_commit_shas('df_commits.csv')
if df is None:
    list_df_commits = [fetch_project_commit_shas(woc_id) for woc_id in my_projects['WoC']]
    df = pd.concat(list_df_commits, ignore_index=True)
    df.to_csv('df_commits.csv', index=False)

df.groupby('project').size()

project
asgr_profound                              631
homalg-project_toricvarieties_project     2445
lanl_elements                              638
ldbc_sigmod2014-pc                         536
libatoms_quip                            11907
nanocomp_meep                             9050
opticspy_lightpipes                        341
ph-rast_bmgarch                            504
wiris_moodle-filter_wiris                 1505
wurmlab_sequenceserver                    3683
dtype: int64

In [18]:
# --- Step 2: Get commit content in batches ---
# Chunks stay under 50 commits with a pause between requests, per the assignment's rate-limit guidance.
CHUNK_SIZE = 30
PAUSE_SECONDS = 5
MAX_ATTEMPTS = 5

# Fetch one chunk of commits, retrying with a growing wait when the WoC server times out.
def fetch_commit_chunk_with_retries(chunk):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return woc.get_values_many('commit.tch', chunk)
        except Exception as error:
            if attempt == MAX_ATTEMPTS:
                raise
            wait_seconds = 5 * attempt
            print(f'Attempt {attempt} failed ({type(error).__name__}); retrying in {wait_seconds}s')
            time.sleep(wait_seconds)

# Each finished chunk is appended here so an interrupted run resumes instead of starting over.
CHECKPOINT_PATH = 'commit_data_checkpoint.jsonl'

# Load commits already fetched by earlier runs.
def load_checkpointed_commits(path):
    if not os.path.exists(path):
        return []
    # Type inference is off so sha1s and timezones like "+0900" stay strings.
    checkpointed = pd.read_json(path, lines=True, dtype=False, convert_dates=False)

    # A resumed run can append commits that are already saved, so keep one row per sha.
    checkpointed = checkpointed.drop_duplicates(subset='commit')
    return checkpointed.to_dict('records')

# Flatten a raw WoC commit record into a dict of named fields.
def flatten_commit(commit_sha, commit):
    # commit = [tree, [parents], [author, unixtime, tz], [committer, unixtime, tz], message]
    return {
        'commit': commit_sha,
        'tree': commit[0],
        'parent': list(commit[1]),
        'author': commit[2][0],
        'author_time': int(commit[2][1]),
        'author_tz': commit[2][2],
        'committer': commit[3][0],
        'committer_time': int(commit[3][1]),
        'committer_tz': commit[3][2],
        'message': commit[4],
    }

commit_data = load_checkpointed_commits(CHECKPOINT_PATH)
already_fetched = {row['commit'] for row in commit_data}
unique_shas = df['sha1'].drop_duplicates().to_list()
remaining_shas = [sha for sha in unique_shas if sha not in already_fetched]
print(f'{len(already_fetched)} commits loaded from checkpoint; {len(remaining_shas)} to fetch')

chunks = [remaining_shas[x:x + CHUNK_SIZE] for x in range(0, len(remaining_shas), CHUNK_SIZE)]
failed_shas = []

for chunk in tqdm(chunks):
    res, err = fetch_commit_chunk_with_retries(chunk)
    res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation
    time.sleep(PAUSE_SECONDS)

    if err:
        print('Got Errors', err)
        failed_shas.extend(err.keys())

    chunk_rows = [flatten_commit(commit_sha, commit) for commit_sha, commit in res.items()]
    commit_data.extend(chunk_rows)
    pd.DataFrame(chunk_rows).to_json(CHECKPOINT_PATH, orient='records', lines=True, mode='a')

print(f'Retrieved {len(commit_data)} of {len(unique_shas)} commits; {len(failed_shas)} failed')

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

31183 commits loaded from checkpoint; 57 to fetch


 50%|█████     | 1/2 [00:05<00:05,  5.10s/it]

Got Errors {'02fc466c09068b8aa412874cc28efef90aa294a1': 'Key 02fc466c09068b8aa412874cc28efef90aa294a1 not found in /da5_fast/All.sha1c/commit_2.tch', '0bfd02d31775423ba3f639bcc5204b8f418e306c': 'Key 0bfd02d31775423ba3f639bcc5204b8f418e306c not found in /da5_fast/All.sha1c/commit_11.tch', '14b5c80c81083e298df5435503b95fa395be1425': 'Key 14b5c80c81083e298df5435503b95fa395be1425 not found in /da5_fast/All.sha1c/commit_20.tch', '1b4eba874d9b8f629e50e403a2a966ede9f30b90': 'Key 1b4eba874d9b8f629e50e403a2a966ede9f30b90 not found in /da5_fast/All.sha1c/commit_27.tch', '26e148f260ae4e74dfa6733a092b50c960d1d8c8': 'Key 26e148f260ae4e74dfa6733a092b50c960d1d8c8 not found in /da5_fast/All.sha1c/commit_38.tch', '29047ec8a3370bf79787a07447e76c913d3a2039': 'Key 29047ec8a3370bf79787a07447e76c913d3a2039 not found in /da5_fast/All.sha1c/commit_41.tch', '3b8bdba974cd7b24767fa32a57e5d70e902b3f3e': 'Key 3b8bdba974cd7b24767fa32a57e5d70e902b3f3e not found in /da5_fast/All.sha1c/commit_59.tch', '3d0eac2e891e756

100%|██████████| 2/2 [00:10<00:00,  5.11s/it]

Got Errors {'22fbc446f1cdf13d2976fb97db7781fefcb4f54d': 'Key 22fbc446f1cdf13d2976fb97db7781fefcb4f54d not found in /da5_fast/All.sha1c/commit_34.tch', '2a2fa6efbd5f1585d4a542c3f3da2a8ee10b2617': 'Key 2a2fa6efbd5f1585d4a542c3f3da2a8ee10b2617 not found in /da5_fast/All.sha1c/commit_42.tch', '31f9b23d6569b71e559e821492ba3ed47ec41f72': 'Key 31f9b23d6569b71e559e821492ba3ed47ec41f72 not found in /da5_fast/All.sha1c/commit_49.tch', '339dc23c70b0f6c6052b933e0aa49a95a93cd442': 'Key 339dc23c70b0f6c6052b933e0aa49a95a93cd442 not found in /da5_fast/All.sha1c/commit_51.tch', '344ef53e05f4cf8d3ae76a3f0d4796fb242c7864': 'Key 344ef53e05f4cf8d3ae76a3f0d4796fb242c7864 not found in /da5_fast/All.sha1c/commit_52.tch', '479a5cd15d07d7bcb6472402d5c5369ee272eda5': 'Key 479a5cd15d07d7bcb6472402d5c5369ee272eda5 not found in /da5_fast/All.sha1c/commit_71.tch', '493554512aa7c3215431df1af7cbd0f520720e6a': 'Key 493554512aa7c3215431df1af7cbd0f520720e6a not found in /da5_fast/All.sha1c/commit_73.tch', '55910aedbbd93b

,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00024f871cba5bd4b07d5d642ecbc482bab0044a,ba12365932ae1302065b0f80c11ee0ee48c56cbe,[4363c49a7ffaed436204cf64846d97941c2a1243],nb326 <nb326@9aeadc09-fa51-0410-b5d8-e5c65b1e4...,1278432690,+0000,nb326 <nb326@9aeadc09-fa51-0410-b5d8-e5c65b1e4...,1278432690,+0000,Add KEdf_radial analysis (r-dep KE distributio...,00024f871cba5bd4b07d5d642ecbc482bab0044a,libatoms_quip
1,0002ca6f185eba1452374f065a357072827ef08c,1367b548908718d0b7a7fde80f02947f5c85e2d0,[d6c153e26d9986d85eb61c193d38aaf06ea39624],jrk33 <jrk33@9aeadc09-fa51-0410-b5d8-e5c65b1e4...,1243505835,+0000,jrk33 <jrk33@9aeadc09-fa51-0410-b5d8-e5c65b1e4...,1243505835,+0000,Fixed failing test: ForceMixing -> FM\n\ngit-s...,0002ca6f185eba1452374f065a357072827ef08c,libatoms_quip


In [19]:
# --- Step 3: Validate that every project's commits were retrieved ---
expected_counts = df.groupby('project').size().rename('p2c_commits')
retrieved_counts = df_commit_data.groupby('project').size().rename('retrieved_commits')
pd.concat([expected_counts, retrieved_counts], axis=1)

,p2c_commits,retrieved_commits
project,,
asgr_profound,631,631
homalg-project_toricvarieties_project,2445,4890
lanl_elements,638,1276
ldbc_sigmod2014-pc,536,1072
libatoms_quip,11907,13178
nanocomp_meep,9050,13088
opticspy_lightpipes,341,341
ph-rast_bmgarch,504,1008
wiris_moodle-filter_wiris,1505,1476


In [20]:
# --- Step 4: Save the project summary CSV ---
summary_columns = {
    'project': 'project_wocid',
    'commit': 'commit_sha1',
    'author': 'author',
    'author_time': 'time',
    'message': 'commit message',
}
dfinf = df_commit_data[list(summary_columns)].rename(columns=summary_columns)
dfinf = dfinf.sort_values(['project_wocid', 'time']).reset_index(drop=True)

dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';')
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,asgr_profound,94e0e00a43c16de98397c711968f048e18d7e68b,Aaron Robotham <aaron.robotham@uwa.edu.au>,1501848698,First ProFound package commit\n


In [21]:
# --- Step 5: WoC stats per project ---
# Convert a unix timestamp to a readable UTC date string.
def format_unix_time_as_date(unix_time):
    return datetime.fromtimestamp(unix_time, tz=timezone.utc).strftime('%Y-%m-%d')

grouped_commits = dfinf.groupby('project_wocid')
woc_stats = pd.DataFrame({
    'ncommits': grouped_commits['commit_sha1'].nunique(),
    'nauthors': grouped_commits['author'].nunique(),
    'from': grouped_commits['time'].min(),
    'to': grouped_commits['time'].max(),
})
woc_stats['from_date'] = woc_stats['from'].apply(format_unix_time_as_date)
woc_stats['to_date'] = woc_stats['to'].apply(format_unix_time_as_date)
woc_stats

,ncommits,nauthors,from,to,from_date,to_date
project_wocid,,,,,,
asgr_profound,631,5,1501848698,1762915006,2017-08-04,2025-11-12
homalg-project_toricvarieties_project,2445,14,1433770389,1736792798,2015-06-08,2025-01-13
lanl_elements,638,34,1595368339,1777314688,2020-07-21,2026-04-27
ldbc_sigmod2014-pc,536,10,1580996984,1748644377,2020-02-06,2025-05-30
libatoms_quip,11907,125,1239384869,1761944464,2009-04-10,2025-10-31
nanocomp_meep,9022,174,1045049250,1776398803,2003-02-12,2026-04-17
opticspy_lightpipes,341,13,1447922910,1725499021,2015-11-19,2024-09-05
ph-rast_bmgarch,504,8,1557274350,1730405362,2019-05-08,2024-10-31
wiris_moodle-filter_wiris,1476,39,1459764927,1762342603,2016-04-04,2025-11-05


In [22]:
# --- Step 6: GitHub stats per project ---
# Set GITHUB_TOKEN to avoid the 60 requests/hour unauthenticated limit.
github_token = os.environ.get('GITHUB_TOKEN')
github_headers = {'Accept': 'application/vnd.github+json'}
if github_token:
    github_headers['Authorization'] = f'Bearer {github_token}'

# Turn a GitHub URL like https://github.com/owner/repo into "owner/repo".
def extract_owner_and_repo(github_url):
    return github_url.rstrip('/').removeprefix('https://github.com/')

# Fetch star count, fork count, and latest default-branch commit date for one repo.
def fetch_github_repo_stats(github_url):
    owner_and_repo = extract_owner_and_repo(github_url)
    repo_api_url = f'https://api.github.com/repos/{owner_and_repo}'

    repo_response = requests.get(repo_api_url, headers=github_headers)
    repo_response.raise_for_status()
    repo_info = repo_response.json()

    latest_commit_response = requests.get(
        f'{repo_api_url}/commits',
        headers=github_headers,
        params={'per_page': 1},
    )
    latest_commit_response.raise_for_status()
    latest_commit = latest_commit_response.json()[0]
    last_commit_date = latest_commit['commit']['committer']['date'][:10]

    return {
        'nstars': repo_info['stargazers_count'],
        'nforks': repo_info['forks_count'],
        'lastGHCommitDate': last_commit_date,
    }

github_rows = []
for _, row in my_projects.iterrows():
    repo_stats = fetch_github_repo_stats(row['GH'])
    repo_stats['project_wocid'] = row['WoC']
    github_rows.append(repo_stats)
    time.sleep(PAUSE_SECONDS)

github_stats = pd.DataFrame(github_rows).set_index('project_wocid')
github_stats

,nstars,nforks,lastGHCommitDate
project_wocid,,,
libatoms_quip,403,131,2026-05-12
ldbc_sigmod2014-pc,3,0,2025-05-30
wurmlab_sequenceserver,302,121,2026-05-11
lanl_elements,24,15,2026-09-22
homalg-project_toricvarieties_project,8,5,2025-01-14
ph-rast_bmgarch,18,11,2026-07-07
nanocomp_meep,1778,811,2026-09-27
wiris_moodle-filter_wiris,17,19,2026-05-29
asgr_profound,51,8,2026-09-26


In [23]:
# --- Step 7: Combined Part 1 stats ---
part1_stats = woc_stats.join(github_stats)
part1_stats.to_csv(yournetid + '_part1_stats.csv', sep=';')
part1_stats

,ncommits,nauthors,from,to,from_date,to_date,nstars,nforks,lastGHCommitDate
project_wocid,,,,,,,,,
asgr_profound,631,5,1501848698,1762915006,2017-08-04,2025-11-12,51,8,2026-09-26
homalg-project_toricvarieties_project,2445,14,1433770389,1736792798,2015-06-08,2025-01-13,8,5,2025-01-14
lanl_elements,638,34,1595368339,1777314688,2020-07-21,2026-04-27,24,15,2026-09-22
ldbc_sigmod2014-pc,536,10,1580996984,1748644377,2020-02-06,2025-05-30,3,0,2025-05-30
libatoms_quip,11907,125,1239384869,1761944464,2009-04-10,2025-10-31,403,131,2026-05-12
nanocomp_meep,9022,174,1045049250,1776398803,2003-02-12,2026-04-17,1778,811,2026-09-27
opticspy_lightpipes,341,13,1447922910,1725499021,2015-11-19,2024-09-05,313,67,2026-02-26
ph-rast_bmgarch,504,8,1557274350,1730405362,2019-05-08,2024-10-31,18,11,2026-07-07
wiris_moodle-filter_wiris,1476,39,1459764927,1762342603,2016-04-04,2025-11-05,17,19,2026-05-29


WOC_RESULTS_PLACEHOLDER

GITHUB_RESULTS_PLACEHOLDER

# Make sure you check in to your fork not just the notebook but also the csv files!!!